<a href="https://colab.research.google.com/github/Madiiinaa/Python/blob/main/%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Вариант 10**

**Уровень 4**

  **Задача 25 (паттерн Factory).** Школа регулярно пополняется разными типами участников — учениками, учителями, администраторами. Создавать их напрямую через конструкторы неудобно: код разрастается, легко ошибиться. Создайте класс `MemberFactory` с методом `create(kind, **kwargs)`, который по строке `\"pupil\"`, `\"teacher\"` или `\"admin\"` создаёт соответствующий объект и возвращает его. Неизвестный тип должен вызывать `ValueError`. Продемонстрируйте работу фабрики: создайте по одному участнику каждого типа через фабрику. Объясните, почему фабрика удобнее прямого вызова конструкторов и как этот паттерн связан с принципом Open/Closed (см. задачу 35).

In [ ]:
class Pupil:
    def __init__(self, name, grade): self.name, self.grade = name, grade
    def __repr__(self): return f"Pupil({self.name}, {self.grade})"

class Teacher:
    def __init__(self, name, subject): self.name, self.subject = name, subject
    def __repr__(self): return f"Teacher({self.name}, {self.subject})"

class Admin:
    def __init__(self, name, position): self.name, self.position = name, position
    def __repr__(self): return f"Admin({self.name}, {self.position})"

class MemberFactory:
    _registry = {"pupil": Pupil, "teacher": Teacher, "admin": Admin}

    @classmethod
    def create(cls, kind, **kwargs):
        if kind not in cls._registry:
            raise ValueError(f"Неизвестный тип участника: {kind}")
        return cls._registry[kind](**kwargs)

print(MemberFactory.create("pupil", name="Мадина", grade="10А"))
print(MemberFactory.create("teacher", name="Марина", subject="Информатика"))
print(MemberFactory.create("admin", name="Мария", position="Директор"))

try:
    MemberFactory.create("robot", name="X")
except ValueError as e:
    print("ValueError:", e)

# Фабрика удобнее прямых конструкторов: клиент не знает конкретных классов,
# добавление нового типа = одна строка в реестре (OCP — открыто для расширения,
# закрыто для изменения существующего кода).

Pupil(Мадина, 10А)
Teacher(Марина, Информатика)
Admin(Мария, Директор)
ValueError: Неизвестный тип участника: robot


**Задача 26 (паттерн Observer).** Ученики и родители хотят получать уведомления о появлении новых уроков или изменении расписания. Реализуйте паттерн «Наблюдатель»: класс `School` хранит список подписчиков (`Pupil`) и при добавлении нового урока уведомляет всех через метод `notify()`. Продемонстрируйте, что при добавлении урока все ученики получают уведомление. Дополнительно реализуйте `unsubscribe()` и покажите, что после отписки ученик больше не получает уведомлений. Объясните, где в реальных проектах используется этот паттерн (GUI, событийные системы, мессенджеры, pub/sub) и как он помогает избежать жёсткой связи между объектами.


In [ ]:
class Pupil:
    def __init__(self, name): self.name = name
    def notify(self, message): print(f"{self.name} получил уведомление: {message}")

class School:
    def __init__(self): self._subscribers = []

    def subscribe(self, pupil): self._subscribers.append(pupil)
    def unsubscribe(self, pupil):
        if pupil in self._subscribers:
            self._subscribers.remove(pupil)

    def add_lesson(self, lesson_name):
        print(f"Добавлен урок: {lesson_name}")
        for p in self._subscribers:
            p.notify(f"новый урок {lesson_name}")

school = School()
madina, olga = Pupil("Мадина"), Pupil("Ольга")
school.subscribe(madina); school.subscribe(olga)

school.add_lesson("Физика")
school.unsubscribe(olga)
school.add_lesson("Химия")

# Observer используется в GUI, pub/sub, мессенджерах, event-bus.
# Он убирает жёсткую связь: школа не знает, кто именно слушает —
# достаточно, что у подписчика есть notify().

Добавлен урок: Физика
Мадина получил уведомление: новый урок Физика
Ольга получил уведомление: новый урок Физика
Добавлен урок: Химия
Мадина получил уведомление: новый урок Химия


**Задача 27 (принцип SOLID — SRP).** Класс `School` со временем разросся: он и хранит учеников, и ведёт расписание, и рассылает уведомления, и формирует отчёты. Это нарушает принцип единственной ответственности (SRP) — у класса должна быть одна причина для изменения. Разбейте класс `School` на несколько классов: `PupilStorage` (хранение учеников и поиск по имени), `ScheduleService` (ведение расписания, запись на уроки), `NotificationService` (уведомления). Продемонстрируйте работу собранных сервисов на сценарии «ученик записался на урок». Объясните, почему это лучше одного большого класса, как понять, что у класса «одна причина для изменения», и приведите примеры «классов-богов» из реальной жизни.


In [ ]:
class Pupil:
    def __init__(self, name): self.name = name

class PupilStorage:
    def __init__(self): self._pupils = []
    def add(self, pupil): self._pupils.append(pupil)
    def find(self, name): return next((p for p in self._pupils if p.name == name), None)

class ScheduleService:
    def __init__(self): self._schedule = {}
    def add_lesson(self, lesson): self._schedule[lesson] = []
    def enroll(self, pupil, lesson):
        if lesson not in self._schedule:
            raise ValueError(f"Урок {lesson} не найден")
        self._schedule[lesson].append(pupil)

class NotificationService:
    def notify(self, pupil, message): print(f" {pupil.name}, {message}")

# Сценарий: ученик записался на урок
storage, schedule, notifier = PupilStorage(), ScheduleService(), NotificationService()
madina = Pupil("Мадина")
storage.add(madina)
schedule.add_lesson("Математика")
schedule.enroll(madina, "Математика")
notifier.notify(madina, "вы записаны на Математику")

# У каждого класса теперь ОДНА причина для изменения:
# - PupilStorage — правила хранения/поиска
# - ScheduleService — правила расписания
# - NotificationService — способы уведомления
# "Класс-бог" — типичный антипаттерн: School, UserManager, Application и т.п.

 Мадина, вы записаны на Математику


**Задача 28 (композиция vs наследование).** Ученики профильных классов (`ProfilePupil`) похожи на обычных учеников, но имеют дополнительные поля — профиль обучения, углублённые предметы. Реализуйте одну и ту же задачу двумя способами: через **наследование** (`ProfilePupil(Pupil)` — переопределяет конструктор и добавляет свои поля) и через **композицию** (`ProfilePupil` содержит объект `Pupil` внутри себя как поле `self.pupil`). Сравните плюсы и минусы каждого подхода: наследование даёт переиспользование кода, но создаёт жёсткую связь; композиция гибче, но требует больше кода. Объясните, когда что выбирать, и приведите примеры из реальных проектов и стандартной библиотеки Python.


In [ ]:
class Pupil:
    def __init__(self, name, grade): self.name, self.grade = name, grade
    def info(self): return f"{self.name} ({self.grade})"

# --- Вариант 1: наследование ---
class ProfilePupilInherit(Pupil):
    def __init__(self, name, grade, profile, subjects):
        super().__init__(name, grade)
        self.profile, self.subjects = profile, subjects
    def info(self): return super().info() + f" — профиль {self.profile}"

# --- Вариант 2: композиция ---
class ProfilePupilCompose:
    def __init__(self, pupil, profile, subjects):
        self.pupil, self.profile, self.subjects = pupil, profile, subjects
    @property
    def name(self): return self.pupil.name
    @property
    def grade(self): return self.pupil.grade
    def info(self): return self.pupil.info() + f" — профиль {self.profile}"

a = ProfilePupilInherit("Мадина", "10Б", "физмат", ["Физика", "Математика"])
b = ProfilePupilCompose(Pupil("Мадина", "10Б"), "физмат", ["Физика", "Математика"])
print(a.info()); print(b.info())

# Наследование: переиспользование кода, но жёсткая связь (is-a).
# Композиция: гибче (можно подменить Pupil, обернуть, тестировать), но больше кода.
# В стандартной библиотеке: io.BufferedWriter (композиция над RawIOBase),
# collections.OrderedDict (наследование от dict) — разные подходы по ситуации.

Мадина (10Б) — профиль физмат
Мадина (10Б) — профиль физмат


  **Задача 29 (копирование объектов).** В школе нужно создавать копии личных дел учеников — например, для архивного хранения. При этом у ученика есть вложенный объект `Backpack`, и важно понимать, копируется ли он вместе с учеником или остаётся общим. Реализуйте в классе `Pupil` методы `__copy__` и `__deepcopy__`. Создайте ученика с вложенным объектом `Backpack` и продемонстрируйте разницу между поверхностным (`copy.copy`) и глубоким (`copy.deepcopy`) копированием: при поверхностном — портфель общий (`shallow.backpack is pupil.backpack` → `True`), при глубоком — свой (`deep.backpack is pupil.backpack` → `False`). Объясните, когда какой тип копирования использовать и что делать с циклическими ссылками.


In [ ]:
import copy

class Backpack:
    def __init__(self): self.items = ["учебник"]
    def __repr__(self): return f"Backpack({self.items})"

class Pupil:
    def __init__(self, name, backpack): self.name, self.backpack = name, backpack

    def __copy__(self):
        # поверхностная копия: новый Pupil, но backpack тот же
        return Pupil(self.name, self.backpack)

    def __deepcopy__(self, memo):
        # глубокая копия: копируем и backpack
        new_backpack = copy.deepcopy(self.backpack, memo)
        return Pupil(self.name, new_backpack)

pupil = Pupil("Мадина", Backpack())
shallow = copy.copy(pupil)
deep = copy.deepcopy(pupil)

print("shallow backpack is pupil backpack:", shallow.backpack is pupil.backpack)  # True
print("   deep backpack is pupil backpack:", deep.backpack is pupil.backpack)     # False

# Поверхностное копирование — когда вложенные объекты можно делить.
# Глубокое — когда нужна полная независимость.
# При циклических ссылках deepcopy использует memo, чтобы не зациклиться.

shallow backpack is pupil backpack: True
   deep backpack is pupil backpack: False


**Задача 30 (утиная типизация).** В школе есть разные объекты, у которых есть метод `describe()` — ученики, учителя, уроки, а может быть, и что-то ещё, что появится в будущем. Напишите функцию `describe_member(member)`, которая вызывает метод `describe()` у любого объекта. Продемонстрируйте, что функция работает не только с `Pupil`, но и с любым другим классом, у которого есть `describe()` (например, классом `Lesson`), — без наследования от общего родителя. Объясните принцип утиной типизации: «если это выглядит как утка, плавает как утка и крякает как утка — это утка». Расскажите, как этот принцип отличает Python от языков со строгой типизацией (Java, C++) и какие у него минусы (ошибка видна только во время выполнения).\n",


In [ ]:
class Pupil:
    def describe(self): return "Ученик"

class Teacher:
    def describe(self): return "Учитель"

class Lesson:
    def describe(self): return "Урок"

def describe_member(member):
    print(member.describe())

for obj in (Pupil(), Teacher(), Lesson()):
    describe_member(obj)

# Утиная типизация: важен не тип, а наличие метода.
# Отличие от Java/C++ — там потребовался бы общий интерфейс/абстрактный класс.
# Минус: ошибка (нет метода) видна только в рантайме — статические анализаторы
# или typing.Protocol помогают поймать такие проблемы заранее.

Ученик
Учитель
Урок


**Уровень 5**

**Задача 31 (итерация по объекту: `__iter__` и `__next__`).** В школе есть класс `Grade`, и хочется перебирать учеников в нём в цикле `for` — так же удобно, как список. Сейчас `for pupil in grade:` не работает. Реализуйте в классе `Grade` магические методы `__iter__` и `__next__` так, чтобы по классу можно было итерироваться **несколько раз подряд** (это важно: `__iter__` должен возвращать отдельный объект-итератор или генератор, а **не** `self`, иначе повторная итерация сломается). Создайте класс с несколькими учениками и переберите их в цикле `for` дважды подряд. Объясните, чем отличается итератор от итерируемого объекта, зачем нужны оба метода и почему `list(grade)` должен возвращать список всех учеников.


In [ ]:
class Grade:
    def __init__(self, pupils): self.pupils = pupils

    def __iter__(self):
        # Возвращаем НОВЫЙ итератор, а не self — иначе 2-й for сломается
        return iter(self.pupils)

g = Grade(["Мадина", "Мария", "Пётр"])
for p in g: print(p)
for p in g: print("снова:", p)   # работает, потому что iter() создаёт новый итератор

print(list(g))

# Итерируемый объект — тот, у кого есть __iter__ (может порождать итераторы).
# Итератор — тот, у кого есть __next__ и он "одноразовый".
# Если __iter__ возвращает self, второй for ничего не выведет — типичная ошибка.

Мадина
Мария
Пётр
снова: Мадина
снова: Мария
снова: Пётр
['Мадина', 'Мария', 'Пётр']


**Задача 32 (индексация объекта: `__getitem__` и `__setitem__`).** В классе ученики должны быть доступны по индексу — как элементы списка: `grade[0]` возвращает первого ученика, `grade[1] = pupil` заменяет второго. Реализуйте в классе `Grade` магические методы `__getitem__` и `__setitem__`. Продемонстрируйте, что теперь работают: `grade[0]` — получение ученика по индексу, `grade[1] = new_pupil` — замена ученика, срезы `grade[0:2]` — получение подсписка, `for pupil in grade` — итерация (за счёт `__getitem__`, даже без `__iter__`). Дополнительно реализуйте `__delitem__` (удаление по индексу) и `__contains__` (проверка `pupil in grade`). Объясните, как `__getitem__` делает объект «похожим» на список или словарь и почему важно, чтобы он выбрасывал именно `IndexError`, а не какое-то своё исключение.


In [ ]:
class Grade:
    def __init__(self, pupils): self.pupils = pupils
    def __getitem__(self, i): return self.pupils[i]      # индекс и срез
    def __setitem__(self, i, v): self.pupils[i] = v
    def __delitem__(self, i): del self.pupils[i]
    def __contains__(self, item): return item in self.pupils

g = Grade(["Мария", "Мария", "Пётр"])
print(g[0]); print(g[0:2])

g[1] = "Ольга"
print(g.pupils)

del g[0]
print(g.pupils)

print("Пётр" in g)

# __getitem__ делает объект похожим на список: индекс, срез, for,
# даже без __iter__. IndexError обязателен — иначе for сломается (нужно
# знать, когда останавливаться).

Мария
['Мария', 'Мария']
['Мария', 'Ольга', 'Пётр']
['Ольга', 'Пётр']
True


**Задача 33 (контекстный менеджер: `__enter__` и `__exit__`).** В школе есть операции, которые нужно выполнять «аккуратно»: например, при работе с базой данных учеников её нужно открыть в начале и обязательно закрыть в конце — даже если произошла ошибка. Реализуйте класс `PupilDatabase` как контекстный менеджер: методы `__enter__` (открывает базу и возвращает её) и `__exit__` (закрывает базу, даже если было исключение). Продемонстрируйте использование через `with PupilDatabase() as db:`. Дополнительно покажите, что происходит, если внутри `with` возникнет исключение, — база всё равно закроется. Объясните, зачем нужны контекстные менеджеры, как они связаны с конструкцией `try/finally`, что делают аргументы `exc_type`, `exc_value`, `traceback` в `__exit__` и что нужно вернуть, чтобы «проглотить» исключение.


In [ ]:
class PupilDatabase:
    def __enter__(self):
        print("База открыта")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("База закрыта")
        if exc_type is not None:
            print(f"Было исключение: {exc_value}")
        return False   # False → исключение пойдёт дальше; True → проглотить

with PupilDatabase() as db:
    print("Работаем с базой")

try:
    with PupilDatabase() as db:
        raise RuntimeError("Ошибка внутри with")
except RuntimeError as e:
    print("Поймано снаружи:", e)

# __exit__ всегда вызывается — как finally.
# exc_type/exc_value/traceback = None, если исключения не было.
# Вернуть True — «проглотить» исключение (полезно для retry-логики).

База открыта
Работаем с базой
База закрыта
База открыта
База закрыта
Было исключение: Ошибка внутри with
Поймано снаружи: Ошибка внутри with


**Задача 34 (паттерн Strategy + `super()`).** В школе есть разные способы выставления итоговой оценки: средняя арифметическая, средняя взвешенная (с учётом веса контрольных) и «щадящая» (для отстающих — округление вверх при спорной оценке). Реализуйте паттерн «Стратегия»: базовый класс `GradeStrategy` с методом `calculate(marks)`, и три наследника — `AverageStrategy`, `WeightedStrategy`, `LenientStrategy`. Каждый наследник переопределяет `calculate`. Дополнительно используйте `super()` в классе `LenientStrategy`, чтобы вызвать реализацию `AverageStrategy` и применить округление вверх. Продемонстрируйте работу всех трёх стратегий через единый интерфейс. Объясните, что такое паттерн «Стратегия», зачем он нужен, как он убирает длинные цепочки `if/elif` и как `super()` помогает переиспользовать код родителя.


In [ ]:
class GradeStrategy:
    def calculate(self, marks): raise NotImplementedError

class AverageStrategy(GradeStrategy):
    def calculate(self, marks): return sum(marks) / len(marks)

class WeightedStrategy(GradeStrategy):
    def calculate(self, marks):
        weights = [2 if m >= 4 else 1 for m in marks]
        return sum(m * w for m, w in zip(marks, weights)) / sum(weights)

class LenientStrategy(AverageStrategy):
    def calculate(self, marks):
        avg = super().calculate(marks)     # переиспользуем среднее
        return min(5, int(avg) + (1 if avg % 1 >= 0.5 else 0))

marks = [4, 5, 3, 4]
for s in (AverageStrategy(), WeightedStrategy(), LenientStrategy()):
    print(type(s).__name__, "->", s.calculate(marks))

# Strategy убирает длинные if/elif: нужен новый алгоритм —
# добавляем новый класс, код клиента не меняется.
# super() в LenientStrategy — пример переиспользования поведения родителя.

AverageStrategy -> 4.0
WeightedStrategy -> 4.142857142857143
LenientStrategy -> 4


**Задача 35 (принципы OCP и LSP из SOLID).** В школе появился новый тип участника — психолог (`Psychologist`). Если добавлять его через изменение базового класса `SchoolItem`, это нарушит принцип **OCP (Open/Closed Principle)** — классы должны быть открыты для расширения, но закрыты для изменения. Дополнительно нужно проверить принцип **LSP (Liskov Substitution Principle)** — объект наследника должен без проблем заменять объект родителя. Реализуйте: базовый класс `SchoolItem` с методом `get_status()`; наследников `Pupil`, `Teacher`; функцию `process_item(item: SchoolItem)`, которая работает с любым участником через родительский интерфейс. Добавьте `Psychologist` **без изменения** существующего кода — это и есть OCP. Затем проверьте, что `process_item` одинаково хорошо работает со всеми тремя типами, — это LSP. Объясните оба принципа и приведите примеры их нарушения (классический пример нарушения LSP: «квадрат — не прямоугольник», если сеттеры ширины и высоты меняют оба измерения).

In [ ]:
class SchoolItem:
    def get_status(self): return "Участник школы"

class Pupil(SchoolItem):
    def get_status(self): return "Ученик"

class Teacher(SchoolItem):
    def get_status(self): return "Учитель"

# Новый тип — БЕЗ изменения существующего кода (OCP)
class Psychologist(SchoolItem):
    def get_status(self): return "Психолог"

def process_item(item: SchoolItem):
    print(item.get_status())

for item in (Pupil(), Teacher(), Psychologist()):
    process_item(item)

# OCP: открыто для расширения (новые наследники), закрыто для изменения
# (код process_item не трогаем).
# LSP: наследник должен работать там, где ожидается родитель.
# Классическое нарушение: Square(Rectangle) — set_width меняет обе стороны,
# и клиент, ожидающий прямоугольник, ломается.

Ученик
Учитель
Психолог


**Уровень 6**

**Задача 36 (объект как функция: `__call__`).** В школе есть служебный объект `AverageCalculator`, который должен уметь считать средний балл по списку оценок. Удобно, чтобы его можно было вызывать как функцию: `calculator(marks)`, а не `calculator.calculate(marks)`. Реализуйте в классе `AverageCalculator` магический метод `__call__(self, marks)`, который возвращает средний балл (например, `sum(marks) / len(marks)`). Продемонстрируйте, что объект можно вызывать как функцию: `calc = AverageCalculator(); calc([4, 5, 5, 3])`. Дополнительно покажите, что такой объект можно передать туда, где ожидается функция-колбэк. Объясните, зачем нужен `__call__`, чем он полезен (функторы, декораторы-классы, колбэки с состоянием) и как он связан с понятием «функтор» в других языках.


In [ ]:
class AverageCalculator:
    def __call__(self, marks):
        return sum(marks) / len(marks)

calc = AverageCalculator()
print(calc([5, 5, 5, 3]))

# Объект можно передать туда, где ждут функцию-колбэк
def apply(func, data): return func(data)
print(apply(calc, [3, 4, 5]))

# __call__ превращает объект в "функтор" (functor).
# Полезно, когда у функции есть состояние: счётчики, кэш, конфигурация.

4.5
4.0


**Задача 37 (`__new__` vs `__init__`).** В школе есть класс `Pupil`, и нужно понять, в каком порядке Python создаёт объект. Реализуйте в классе `Pupil` оба метода — `__new__` и `__init__` — и добавьте в каждый отладочный вывод (`print`), чтобы увидеть порядок вызова. Продемонстрируйте, что `__new__` вызывается **до** `__init__` и отвечает за создание объекта, а `__init__` — за его инициализацию. Дополнительно реализуйте `__new__`, который возвращает уже существующий объект (например, для кэширования учеников по имени и классу): два вызова `Pupil(\"Иван Петров\", \"9А\", 4.5)` должны вернуть **один и тот же** объект. Объясните разницу между `__new__` и `__init__`, зачем вообще нужен `__new__` и почему «`__new__` + кэш» легко приводит к странному поведению.

In [ ]:
class Pupil:
    _cache = {}

    def __new__(cls, name, grade, average_mark=None):
        key = (name, grade)
        if key not in cls._cache:
            print("__new__: создаём новый объект")
            cls._cache[key] = super().__new__(cls)
        return cls._cache[key]

    def __init__(self, name, grade, average_mark=None):
        print("__init__: инициализируем")
        self.name, self.grade, self.average_mark = name, grade, average_mark

a = Pupil("Мадина", "10А", 4.5)
b = Pupil("Мадина", "10А", 5.0)
print("один и тот же объект:", a is b)
print("mark:", a.average_mark)

# __new__ создаёт объект (до __init__), __init__ его настраивает.
# Кэш по ключу — вариант Singleton/Flyweight.
# Опасность: __init__ вызывается повторно, перезаписывая поля —
# легко получить странное поведение, поэтому такой приём применять осторожно.

__new__: создаём новый объект
__init__: инициализируем
__init__: инициализируем
один и тот же объект: True
mark: 5.0


**Задача 38 (паттерн Decorator).** В школе есть базовый класс `SchoolItem` с методом `get_info()`. Нужно добавлять к участнику дополнительную информацию — например, «отличник», «староста», «участник олимпиад». При этом нельзя менять сам класс `SchoolItem`. Реализуйте паттерн «Декоратор»: класс `ItemDecorator`, который содержит объект `SchoolItem` внутри себя и имеет тот же метод `get_info()`, но добавляет к результату дополнительный текст. Создайте декораторы `ExcellentDecorator` и `GroupLeaderDecorator`, которые можно комбинировать: `ExcellentDecorator(GroupLeaderDecorator(pupil))`. Продемонстрируйте работу, а также покажите, что смена порядка декораторов меняет порядок меток в выводе. Объясните, чем паттерн «Декоратор» отличается от наследования и почему он удобнее, когда нужно много комбинаций украшений (иначе пришлось бы создавать 2^n подклассов).


In [ ]:
class SchoolItem:
    def __init__(self, name): self.name = name
    def get_info(self): return f"{self.name}"

class ItemDecorator(SchoolItem):
    def __init__(self, item): self.item = item
    def get_info(self): return self.item.get_info()

class ExcellentDecorator(ItemDecorator):
    def get_info(self): return super().get_info() + " [ударник]"

class GroupLeaderDecorator(ItemDecorator):
    def get_info(self): return super().get_info() + " [участник олимпиад]"

pupil = SchoolItem("Мадина")
print(ExcellentDecorator(GroupLeaderDecorator(pupil)).get_info())
print(GroupLeaderDecorator(ExcellentDecorator(pupil)).get_info())

# Декоратор оборачивает объект и добавляет поведение — без наследования
# и без изменения класса. 4 украшения = 16 комбинаций — наследованием было бы
# 16 подклассов, декораторами — 4 класса, комбинируются как угодно.

Мадина [участник олимпиад] [ударник]
Мадина [ударник] [участник олимпиад]


**Задача 39 (паттерн Adapter).** В школе появилась внешняя система электронного журнала, которая возвращает данные об учениках в другом формате — например, в виде словаря `{\"fio\": \"Иван Петров\", \"class\": \"9А\", \"gpa\": 4.5}`. А весь остальной код школы работает с классом `Pupil`, у которого поля `name`, `grade`, `average_mark`. Реализуйте паттерн «Адаптер»: класс `PupilAdapter`, который принимает словарь и предоставляет интерфейс `Pupil` — свойства `name`, `grade`, `average_mark` и метод `info()`. Продемонстрируйте, что адаптер можно использовать там, где ожидается `Pupil` — например, передать его в функцию, которая работает с учеником. Объясните, что такое паттерн «Адаптер», зачем он нужен, чем отличается от Facade и Decorator, и как он помогает интегрировать несовместимые интерфейсы без изменения существующего кода.


In [ ]:
class Pupil:
    def __init__(self, name, grade, average_mark):
        self.name, self.grade, self.average_mark = name, grade, average_mark
    def info(self): return f"{self.name} ({self.grade}), ср. {self.average_mark}"

class PupilAdapter:
    def __init__(self, data: dict):
        self._data = data
    @property
    def name(self): return self._data["fio"]
    @property
    def grade(self): return self._data["class"]
    @property
    def average_mark(self): return self._data["gpa"]
    def info(self): return f"{self.name} ({self.grade}), ср. {self.average_mark}"

def print_pupil(p):   # ожидает интерфейс Pupil
    print(p.info())

print_pupil(Pupil("Мадина", "10А", 4.5))
print_pupil(PupilAdapter({"fio": "Мария", "class": "9Б", "gpa": 4.8}))

# Adapter приводит несовместимый интерфейс к ожидаемому.
# Отличие от Decorator: декоратор добавляет поведение, адаптер — меняет интерфейс.
# Отличие от Facade: фасад упрощает сложную подсистему, адаптер — конкретный объект.

Мадина (10А), ср. 4.5
Мария (9Б), ср. 4.8


**Задача 40 (принципы ISP и DIP из SOLID).** В школе есть базовый класс `SchoolItem`, и постепенно в нём накопились методы, которые нужны не всем участникам: `teach()` (только для учителей), `learn()` (только для учеников), `manage()` (только для администрации). Это нарушает принцип **ISP (Interface Segregation Principle)** — клиент не должен зависеть от методов, которые он не использует. Разделите интерфейс на несколько маленьких: `Teachable`, `Learnable`, `Manageable`, и пусть каждый класс реализует только нужные. Дополнительно проверьте принцип **DIP (Dependency Inversion Principle)** — модули высокого уровня не должны зависеть от модулей низкого уровня; оба должны зависеть от абстракций. Реализуйте класс `LessonService`, который зависит не от конкретных классов (`Pupil`, `Teacher`), а от абстракции `Learnable` (или от `Protocol`). Продемонстрируйте, что можно легко добавить новый тип участника (например, `Psychologist`, реализующий только `Manageable`) без изменения `LessonService`. Объясните оба принципа, приведите примеры их нарушения и расскажите, почему DIP помогает тестировать код (легко подменять зависимости на mock).


In [ ]:
from typing import Protocol

# --- Маленькие интерфейсы (ISP) ---
class Learnable(Protocol):
    def learn(self): ...

class Teachable(Protocol):
    def teach(self): ...

class Manageable(Protocol):
    def manage(self): ...

# --- Участники реализуют только нужное ---
class Pupil:
    def learn(self): return "учится"

class Teacher:
    def teach(self): return "учит"

class Admin:
    def manage(self): return "управляет"

class Psychologist:
    def manage(self): return "консультирует"

# --- DIP: сервис зависит от абстракции Learnable, не от конкретного Pupil ---
class LessonService:
    def __init__(self, learners: list[Learnable]):
        self.learners = learners
    def run(self):
        for l in self.learners:
            print(l.learn())

service = LessonService([Pupil(), Pupil()])
service.run()

# ISP: никто не обязан реализовывать teach/manage, если он только учится.
# DIP: LessonService зависит от абстракции Learnable — легко подменить
# на mock в тестах или на нового участника (например, "стажёр").
# Добавить Psychologist можно без изменения LessonService — он просто не Learnable.

учится
учится
